# PySceneDetect

## Documentación

https://www.scenedetect.com/docs/latest/api

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

<p> 
PySceneDetect es una herramienta para detectar cambios de plano en vídeos (ejemplo), y puede dividir automáticamente el vídeo en clips separados. PySceneDetect es un software gratuito y de código abierto, y dispone de varios métodos de detección para encontrar cortes rápidos y fundidos basados en umbrales.

Sus usos:
</p>

<ul>
    <li>
        División de vídeos domésticos u otras fuentes de vídeo en escenas individuales
    </li>
    <li>
        Detección y eliminación automática de anuncios publicitarios de fuentes de vídeo guardadas en PVR
    </li>
    <li>
        Procesamiento y división de imágenes de cámaras de vigilancia
    </li>
    <li>
        Análisis estadístico de vídeos para encontrar «bucles» adecuados para GIF/cinemagraphs en bucle
    </li>
    <li>
        Análisis académico de películas y vídeos (por ejemplo, búsqueda de la duración media de las tomas)
    </li>
    <ul>
</div>

## Importar librerías

In [27]:
import numpy as np
import pandas as pd
from enum import Enum
from scenedetect import StatsManager,ContentDetector, AdaptiveDetector, SceneManager,detect, split_video_ffmpeg, open_video
from scenedetect.platform import init_logger
from scenedetect.scene_manager import get_scenes_from_cuts, Interpolation, save_images
from scenedetect.backends import AVAILABLE_BACKENDS

In [2]:
# Initializes logging for PySceneDetect. The logger instance used is named ‘pyscenedetect’. By default the logger has no handlers to suppress output. All existing log handlers are replaced every time this function is invoked.
init_logger(log_level=20, show_stdout=False, log_file=None)

In [3]:
VIDEO_PATH = '/root/work/_Videos_De_Clase/test_01.mp4'

: 

In [4]:
OUTPUT_DIR = os.environ['OUTPUT_DIR']

In [5]:
STATS_FILE_PATH = os.environ['STATS_FILE_PATH']

In [6]:
video = open_video(VIDEO_PATH)

In [7]:
# Callback to invoke on the first frame of every new scene detection.
def on_new_scene(frame_img: np.ndarray, frame_num: int):
    print("New scene found at frame %d." % frame_num)

In [8]:
def print_scenes(scene_list: list):
    for i, (start_time, end_time) in enumerate(scene_list, start=1):
        print("Escena {}: {} - {}".format(i, start_time.get_timecode(), end_time.get_timecode()))

In [9]:
class Interpolation(Enum):
    INTERPOLATION_NEAREST = 0
    INTERPOLATION_LINEAR = 1
    INTERPOLATION_CUBIC = 2
    INTERPOLATION_AREA = 3
    INTERPOLATION_LANCZOS4 = 4

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

<p> 
(Teoría)
PySceneDetec cuenta con varios métodos de interpolación de imagenes. La interpolación es la función que se utiliza para interpolar (dividir)  imágenes de máquina.

Al momento de tranformar una imagen, ya sea zoom in o zoom out, para mezclar o rellenar la imagen,
al momento de decidir con que realizar la transformación, se puede escoger entre diferentes interpolaciones:
</p>

<p>
<strong>NEAREST</strong>: Se elige el color del pixel más cercano. Esto hace que no haya ningún calculo en la interpolación, por lo que otorga una alta velocidad de procesamiento, pero produce resultados con menos suavizado (esto puede causar falsos positivos)

Caso de uso: Procesamiento rápido de imágenes.
</p>

<p>
<strong>LINEAR</strong>: Se realiza la ponderación de 4 pixeles vecinos (2x2). Esto empieza agregar calculos en la interpolación, y produce resultados con algo suavizado.

Caso de uso: Procesamiento de imágenes equilibrado entre procesamiento y calidad.
</p>

<p>
<strong>CUBIC</strong>: Se realiza la ponderación de 16 pixeles vecinos (4x4), agregando calculos con polinomios de tercer grado. Esto agrega muchos más calculos, y produce resultados con mayor suavizado.

Caso de uso: Procesamiento de imágenes con alta calidad y procesamiento más lento.
</p>

<p>
<strong>AREA</strong>: Se realiza cuando se necesita hacer la imagen más pequeña. Esto realiza un promedio de todos los pixeles y generá una imagen limpia, sin ruido ni detalles innecesarios.

Caso de uso: Procesamiento de imágenes con alta calidad y procesamiento más lento.
</p>

<p>
<strong>LANCZOS4</strong>: Emplea un filtro de una función 'sinc()' (una función matemática para recontruir señales a partir de muestras), esta función permite obtener resultados más suavizados y fieles a la imagen original.

Caso de uso: Procesamiento de imágenes con muy alta calidad (sensible a cambios menores) a costo del procesamiento más lento.
</p>


<p>
Teoricamente, LANCZOS4 ofrece los mejores resultados, y aunque es más costoso, preferiremos su uso.
</p>


<a href="https://www.scenedetect.com/docs/latest/api/scene_manager.html"> Documentación Interpolación</a>

</div>

#### Resultados por tipo de interpolación (ContentDetector)

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

<p>
Nearest: 18 (27033)
</p>
<p>
Linear: 18 (27033)
</p>
<p>
Cubic: 18 (27033)
</p>
<p>
Area: 18 (27033)
</p>
<p>
Lanczos4: 18 (27033)
</p>
</div>

In [10]:
interpolation = Interpolation.INTERPOLATION_LANCZOS4.value

In [11]:
Interpolation(interpolation)

<Interpolation.INTERPOLATION_LANCZOS4: 4>

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

<p> 
(Teoría)
PySceneDetec cuenta con varios algoritmos para la detección de cambios de escenas en imágenes.
</p>

<p>
<strong>ContentDetector</strong>: Detecta cambios bruscos haciendo un calculo promedio de la diferencia de los pixeles de la imagen. Esto sucede en el espacio de color HSV (Hue, Saturation, Value).

Caso de uso: Detecta muy bien los cambios bruscos en imágenes.
</p>

<p>
<strong>ThresholdDetector</strong>: Detecta cambios en la intensidad global de los pixeles de la imagen, tiene un umbral que indica la intensidad minima para considerar un cambio.

Caso de uso: Detecta cambios en transiciones lentas (como cambios de iluminación) o cambios sutiles en presentaciones.
</p>

<p>
<strong>AdaptiveDetector</strong>: Detecta cambios en la intensidad global de los pixeles de la imagen pero en el espacio de coloers HSV (Hue, Saturation, Value), hace un calculo de la diferencia de los pixeles de la imagen.

Caso de uso: Detecta cambios en transiciones rápidas.
</p>

<p>
<strong>HistogramDetector</strong>: Detecta cambios con la diferencias de histogramas de los pixeles de la imagen, esto se centra en la iluminación Y en el espacio de color YUV.

Caso de uso: Detecta cambios en la estructura y en la exposición presentes en cortes rapidos.
</p>

<p>
<strong>HashDetector</strong>: Detecta cambios utilizando calculo de Hash Perceptual (HPP) de los pixeles de la imagen. Se generá un Hash para cada frame y se compara con el anterior con un umbral de tolerancia.

Caso de uso: Procesamiento de imágenes con muy alta calidad (sensible a cambios menores) a costo del procesamiento más lento.
</p>

<p>
Teoricamente, AdaptiveDetector ofrece la mayoría de parametros para mejorar adaptativamente la detección de escenas, por lo qué, partiremos por ese inicialmente.
</p>

<a href="https://www.scenedetect.com/docs/latest/api/detectors.html"> Documentación de Algoritmos</a>

</div>

In [12]:
ADAPTATIVE_THRESHOLD = 15.0
MIN_SCENE_LEN = 20
WINDOW_WIDTH = 8
MIN_CONTENT_VAL = 12
LUMA_ONLY = True
KERNEL_SIZE = None

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

<p>ADAPTATIVE_THRESHOLD = 15.0 # Un umbral de 15% entre diferencia de pixeles para considerar un cambio.</p>
<p>MIN_SCENE_LEN = 20 # Cada escena debe tener 20 frames o más.</p>
<p>WINDOW_WIDTH = 8 # Cada escena debe ser de 8 frames o más.</p>
<p>MIN_CONTENT_VAL = 12 # Cada escena debe tener 12 pixeles o más, para asegurar un cambio real.</p>
<p>LUMA_ONLY = True # Se enfoca unicamente en la luminosidad, ya que en los videos de clase no habrá muchos cambios de color.</p>
<p>KERNEL_SIZE = None # Permite un ajuste automatico en función de resolución de videos</p>

</div>

In [13]:
scene_manager = SceneManager(stats_manager=StatsManager())
scene_manager.add_detector(AdaptiveDetector(adaptive_threshold=ADAPTATIVE_THRESHOLD, min_scene_len=MIN_SCENE_LEN, window_width=WINDOW_WIDTH, min_content_val=MIN_CONTENT_VAL, luma_only=LUMA_ONLY, kernel_size=KERNEL_SIZE))

In [14]:
# Detect all scenes in video from current position to end.
scene_manager.detect_scenes(video, callback=on_new_scene, show_progress=True)

  Detected: 2 | Progress:   0%|          | 110/27033 [00:00<01:12, 371.87frames/s]New scene found at frame 37.
New scene found at frame 81.
  Detected: 3 | Progress:   9%|▉         | 2418/27033 [00:06<01:00, 408.38frames/s]New scene found at frame 2347.
  Detected: 4 | Progress:  16%|█▌        | 4346/27033 [00:10<00:57, 396.13frames/s]New scene found at frame 4286.
  Detected: 6 | Progress:  45%|████▌     | 12171/27033 [00:30<00:36, 411.58frames/s]New scene found at frame 12111.
New scene found at frame 12160.
  Detected: 7 | Progress:  50%|████▉     | 13397/27033 [00:33<00:34, 395.21frames/s]New scene found at frame 13347.
  Detected: 8 | Progress:  53%|█████▎    | 14456/27033 [00:36<00:31, 399.28frames/s]New scene found at frame 14376.
  Detected: 10 | Progress:  56%|█████▋    | 15268/27033 [00:38<00:29, 401.80frames/s]New scene found at frame 15185.
New scene found at frame 15218.
  Detected: 11 | Progress:  57%|█████▋    | 15430/27033 [00:38<00:29, 393.38frames/s]New scene found at

27033

In [15]:
scene_list = scene_manager.get_scene_list()
print(scene_list)
print_scenes(scene_list=scene_list)

[(00:00:00.000 [frame=0, fps=14.999], 00:00:02.467 [frame=37, fps=14.999]), (00:00:02.467 [frame=37, fps=14.999], 00:00:05.400 [frame=81, fps=14.999]), (00:00:05.400 [frame=81, fps=14.999], 00:02:36.476 [frame=2347, fps=14.999]), (00:02:36.476 [frame=2347, fps=14.999], 00:04:45.750 [frame=4286, fps=14.999]), (00:04:45.750 [frame=4286, fps=14.999], 00:13:27.447 [frame=12111, fps=14.999]), (00:13:27.447 [frame=12111, fps=14.999], 00:13:30.714 [frame=12160, fps=14.999]), (00:13:30.714 [frame=12160, fps=14.999], 00:14:49.851 [frame=13347, fps=14.999]), (00:14:49.851 [frame=13347, fps=14.999], 00:15:58.455 [frame=14376, fps=14.999]), (00:15:58.455 [frame=14376, fps=14.999], 00:16:52.392 [frame=15185, fps=14.999]), (00:16:52.392 [frame=15185, fps=14.999], 00:16:54.592 [frame=15218, fps=14.999]), (00:16:54.592 [frame=15218, fps=14.999], 00:17:04.993 [frame=15374, fps=14.999]), (00:17:04.993 [frame=15374, fps=14.999], 00:21:16.407 [frame=19145, fps=14.999]), (00:21:16.407 [frame=19145, fps=14.

In [16]:
# Save per-frame statistics to disk.
scene_manager.stats_manager.save_to_csv(csv_file=STATS_FILE_PATH)

In [29]:
pd = pd.read_csv(STATS_FILE_PATH)


In [31]:
pd

,Frame Number,Timecode,adaptive_ratio_lum (w=8),content_val,delta_edges,delta_hue,delta_lum,delta_sat
0,2,00:00:00.067,None,0.034017,0.027669,0.000000,0.034017,0.000000
1,3,00:00:00.133,None,0.000000,0.000000,0.000000,0.000000,0.000000
2,4,00:00:00.200,None,0.000000,0.000000,0.000000,0.000000,0.000000
3,5,00:00:00.267,None,0.000000,0.000000,0.000000,0.000000,0.000000
4,6,00:00:00.333,None,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...
27027,27029,00:30:01.971,None,0.058757,0.013835,0.126166,0.058757,0.027479
27028,27030,00:30:02.038,None,0.001085,0.027669,0.000271,0.001085,0.001275
27029,27031,00:30:02.104,None,0.000000,0.000000,0.000000,0.000000,0.000000
27030,27032,00:30:02.171,None,0.000000,0.000000,0.000000,0.000000,0.000000


In [17]:
# Save a set number of images from each scene, given a list of scenes and the associated video/frame source.
save_images(video=video, scene_list=scene_list, output_dir=OUTPUT_DIR, image_extension='webp', encoder_param=80, show_progress=True)

 99%|█████████▊| 68/69 [00:14<00:00,  4.68images/s]
Could not generate all output images.


{0: ['test_01-Scene-001-01.webp',
  'test_01-Scene-001-02.webp',
  'test_01-Scene-001-03.webp'],
 1: ['test_01-Scene-002-01.webp',
  'test_01-Scene-002-02.webp',
  'test_01-Scene-002-03.webp'],
 2: ['test_01-Scene-003-01.webp',
  'test_01-Scene-003-02.webp',
  'test_01-Scene-003-03.webp'],
 3: ['test_01-Scene-004-01.webp',
  'test_01-Scene-004-02.webp',
  'test_01-Scene-004-03.webp'],
 4: ['test_01-Scene-005-01.webp',
  'test_01-Scene-005-02.webp',
  'test_01-Scene-005-03.webp'],
 5: ['test_01-Scene-006-01.webp',
  'test_01-Scene-006-02.webp',
  'test_01-Scene-006-03.webp'],
 6: ['test_01-Scene-007-01.webp',
  'test_01-Scene-007-02.webp',
  'test_01-Scene-007-03.webp'],
 7: ['test_01-Scene-008-01.webp',
  'test_01-Scene-008-02.webp',
  'test_01-Scene-008-03.webp'],
 8: ['test_01-Scene-009-01.webp',
  'test_01-Scene-009-02.webp',
  'test_01-Scene-009-03.webp'],
 9: ['test_01-Scene-010-01.webp',
  'test_01-Scene-010-02.webp',
  'test_01-Scene-010-03.webp'],
 10: ['test_01-Scene-011-01.we

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=8e809eb8-650a-4468-a096-2a2830a4a1e1' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>